# IOAI — 2024 On Site Round Help Bobai — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/IOAI-official/IOAI-2024"
CLONE = "IOAI-2024"
SUBDIR = "On-Site-Round/Help_BOBAI"
WORKDIR = "On-Site-Round/Help_BOBAI/Solution"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

<img src="./figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2024 (불가리아 부르가스), 온사이트 라운드](https://ioai-official.org/bulgaria-2024)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2024/blob/main/On-Site-Round/Help_BOBAI/Solutioin/Help_BOBAI_Solution.ipynb)

# Help BOBAI: 모범답안

## 학습 데이터셋
학습 데이터(`train-dev`)를 불러와 입력 `inputs`, 라벨 `labels` 로 둔다.

In [ ]:
import torch

dataset = torch.load('../training_set/train-dev_dataset_with_labels.pt')
inputs = dataset[:, :, :-1]
labels = dataset[:, :, -1]

# 풀이 (Solution)

베이스라인은 새 라벨(5, 6)을 **무작위로** 배정했다. 이 셀을 자기 풀이로 교체한다 — 베이스라인의 *"You can replace the code below with your solution."* 자리.

**BOBAI 게이팅**: 학습 입력으로 `기존<5→0 / 5→1 / 6→2` 게이트 **KNN(euclidean, k=3)** 을 학습. 추론 시 KNN 이 **신규**면 `which+4`, **기존**이면 동결 `base_clf` 를 쓰는 `clf` 를 만든다.

In [ ]:
import torch
from sklearn.neighbors import KNeighborsClassifier

# 게이트 라벨: 기존<5 → 0, 5 → 1, 6 → 2
gate_labels = torch.where(labels < 5, 0, torch.where(labels == 5, 1, 2)).reshape(-1)
X = inputs.reshape(-1, 768)

# 게이트 KNN (euclidean, k=3)
knn = KNeighborsClassifier(n_neighbors=3, weights='distance')
knn.fit(X, gate_labels)

class SevenWayClassifier():
    def __init__(self):
        base_clf = torch.nn.Linear(768, 5)
        base_clf.load_state_dict(torch.load("../training_set/base_classifier.pth"))
        base_clf.eval()
        self.base_clf = base_clf
    def base_classification(self, input_vector):
        with torch.no_grad():
            preds = torch.softmax(self.base_clf(input_vector.float()), 1)
            return preds.argmax(dim=1).numpy()[0]
    def __call__(self, input_vector):
        which = knn.predict(input_vector)[0]   # 0=기존, 1/2=신규
        if which:
            return which + 4                   # 신규 → 5 또는 6
        return self.base_classification(input_vector)

clf = SevenWayClassifier()

## 추론 및 평가
학습 데이터에 `clf` 를 적용해 macro-F1 로 빠른 점검(sanity check). 실제 점수는 아래 제출 파일(`submission.csv`)을 채점해 확인한다.

In [ ]:
from sklearn.metrics import f1_score
from tqdm import tqdm

def compute_f1(labels, predictions):
    return f1_score(labels, predictions, average='macro')

def inference(clf, input_vectors):
    predictions = []
    for sample in tqdm(input_vectors):
        predictions.append(clf(sample))
    return predictions

predictions = inference(clf, inputs)
print('\n모범답안 F1', compute_f1(labels, predictions))

## 검증 데이터셋
검증셋 예측 → `submission_val.csv` (정답 비공개, 참고용).

In [ ]:
import pandas as pd
import numpy as np

def submission_to_csv(pred, output_fpath="submission.csv"):
    pred = np.array(pred).flatten()
    pd.DataFrame({"ID": np.arange(pred.size), "class": pred}).to_csv(output_fpath, index=False)

eval_inputs = torch.load('./validation_set/eval_dataset.pt')
submission_to_csv(inference(clf, eval_inputs), "submission_val.csv")
print("submission_val.csv 생성 완료")

## 테스트 데이터셋
테스트셋 예측 → 채점 대상 `submission.csv` (ID, class).

In [ ]:
# 이 셀은 변경하지 마세요
test_inputs = torch.load('./test_set/test_dataset.pt')
test_predictions = inference(clf, test_inputs)
submission_to_csv(test_predictions, "submission.csv")
print("submission.csv (test):", len(test_predictions), "행 생성 완료")

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)